In [5]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import concat, lit, col, lpad

In [6]:
def create_spark_session(app_name: str) -> SparkSession:
    spark = (
        SparkSession.builder
        .master("spark://spark-master:7077")
        .appName(app_name)
        .config("spark.sql.extensions", "org.apache.iceberg.spark.extensions.IcebergSparkSessionExtensions")
        .config("spark.sql.catalog.lakehouse_prod","org.apache.iceberg.spark.SparkCatalog")
        .config("spark.sql.catalog.lakehouse_prod.type", "hive")
        .config("spark.sql.catalog.lakehouse_prod.uri","thrift://hive-metastore:9083")
        .config("spark.sql.catalog.lakehouse_prod.warehouse","s3a://lakehouse-prod-bucket/warehouse/")
        .config("spark.sql.catalog.lakehouse_prod.io-impl","org.apache.iceberg.aws.s3.S3FileIO")
        .config("spark.hadoop.fs.s3a.endpoint", "http://minio:9000")
        .config("spark.hadoop.fs.s3a.path.style.access", "true")
        .config("spark.hadoop.fs.s3a.impl", "org.apache.hadoop.fs.s3a.S3AFileSystem")
        .config("spark.hadoop.fs.s3a.connection.ssl.enabled", "false")
        .config("spark.sql.adaptive.enabled", "true")
        .config("spark.serializer", "org.apache.spark.serializer.KryoSerializer")
        .enableHiveSupport()
        .getOrCreate()
    )

    spark.sparkContext.setLogLevel("ERROR")
    print("NOTE: SparkSession created successfully!")

    return spark


app_name = 'Spark-Iceberg-ETL_Bronze-Silver-Test'
spark = create_spark_session(app_name)
spark

NOTE: SparkSession created successfully!


In [8]:
if not spark.catalog.tableExists("bronze_db.bronze_country_iso_profiles_l"):
    print("Table does NOT exist")

Table does NOT exist


26/03/20 17:07:01 ERROR TaskSchedulerImpl: Lost executor 2 on 172.22.0.10: Executor heartbeat timed out after 158287 ms
26/03/20 17:07:01 ERROR Inbox: Ignoring error
java.lang.AssertionError: assertion failed: BlockManager re-registration shouldn't succeed when the executor is lost
	at scala.Predef$.assert(Predef.scala:223)
	at org.apache.spark.storage.BlockManagerMasterEndpoint.org$apache$spark$storage$BlockManagerMasterEndpoint$$register(BlockManagerMasterEndpoint.scala:727)
	at org.apache.spark.storage.BlockManagerMasterEndpoint$$anonfun$receiveAndReply$1.applyOrElse(BlockManagerMasterEndpoint.scala:133)
	at org.apache.spark.rpc.netty.Inbox.$anonfun$process$1(Inbox.scala:103)
	at org.apache.spark.rpc.netty.Inbox.safelyCall(Inbox.scala:213)
	at org.apache.spark.rpc.netty.Inbox.process(Inbox.scala:100)
	at org.apache.spark.rpc.netty.MessageLoop.org$apache$spark$rpc$netty$MessageLoop$$receiveLoop(MessageLoop.scala:75)
	at org.apache.spark.rpc.netty.MessageLoop$$anon$1.run(MessageLoop.s

In [3]:
spark.sql("""
SELECT * FROM lakehouse_prod.bronze_db.bronze_country_iso_profiles LIMIT 5;
""").show()

+----+----+-------+--------------+--------------+--------------------+
|iso2|iso3|iso_num|       country|country_common|       Ingested_Time|
+----+----+-------+--------------+--------------+--------------------+
|  AF| AFG|      4|   Afghanistan|   Afghanistan|2026-03-17 15:07:...|
|  AX| ALA|    248| Aland Islands| Aland Islands|2026-03-17 15:07:...|
|  AL| ALB|      8|       Albania|       Albania|2026-03-17 15:07:...|
|  DZ| DZA|     12|       Algeria|       Algeria|2026-03-17 15:07:...|
|  AS| ASM|     16|American Samoa|American Samoa|2026-03-17 15:07:...|
+----+----+-------+--------------+--------------+--------------------+



In [6]:
spark.sql("""
SELECT * FROM lakehouse_prod.bronze_db.bronze_country_iso_profiles.files;
""").show()

+-------+--------------------+-----------+-------+------------+------------------+--------------------+--------------------+--------------------+----------------+--------------------+--------------------+------------+-------------+------------+-------------+--------------------+
|content|           file_path|file_format|spec_id|record_count|file_size_in_bytes|        column_sizes|        value_counts|   null_value_counts|nan_value_counts|        lower_bounds|        upper_bounds|key_metadata|split_offsets|equality_ids|sort_order_id|    readable_metrics|
+-------+--------------------+-----------+-------+------------+------------------+--------------------+--------------------+--------------------+----------------+--------------------+--------------------+------------+-------------+------------+-------------+--------------------+
|      0|s3a://lakehouse-b...|    PARQUET|      0|         249|              6977|{1 -> 359, 2 -> 6...|{1 -> 249, 2 -> 2...|{1 -> 1, 2 -> 0, ...|              {

In [15]:
spark.sql("""
SELECT * FROM lakehouse_prod.bronze_db.bronze_country_iso_profiles.history;
""").show()

+--------------------+-------------------+---------+-------------------+
|     made_current_at|        snapshot_id|parent_id|is_current_ancestor|
+--------------------+-------------------+---------+-------------------+
|2026-03-16 16:22:...|1226238839559789206|     NULL|               true|
+--------------------+-------------------+---------+-------------------+



In [10]:
spark.sql("""
SELECT * FROM lakehouse_prod.bronze_db.bronze_sp500_profiles.files;
""").toPandas()

,content,file_path,file_format,spec_id,record_count,file_size_in_bytes,column_sizes,value_counts,null_value_counts,nan_value_counts,lower_bounds,upper_bounds,key_metadata,split_offsets,equality_ids,sort_order_id,readable_metrics
0,0,s3a://lakehouse-bronze-bucket/warehouse/bronze...,PARQUET,0,503,721076,"{1: 5515, 2: 2234, 3: 616, 4: 2383, 5: 243, 6:...","{1: 503, 2: 503, 3: 503, 4: 503, 5: 503, 6: 50...","{1: 1, 2: 1, 3: 24, 4: 3, 5: 1, 6: 3, 7: 1, 8:...","{15: 0, 17: 0, 18: 0, 19: 0, 20: 0, 21: 0, 22:...","{1: [49, 32, 66, 101, 99, 116, 111, 110, 32, 6...","{1: [87, 97, 116, 101, 114, 108, 111, 111, 32,...",None,[4],None,0,"((3908, None, None, None, None, None), (78, No..."


In [49]:
spark.sql("""
SELECT * FROM lakehouse_prod.bronze_db.bronze_sp500_profiles.history;
""").show()

+--------------------+-------------------+---------+-------------------+
|     made_current_at|        snapshot_id|parent_id|is_current_ancestor|
+--------------------+-------------------+---------+-------------------+
|2026-03-14 09:51:...|4435125849619209165|     NULL|              false|
|2026-03-16 13:11:...|2168248009461615393|     NULL|              false|
|2026-03-16 15:45:...|4976747531334553245|     NULL|              false|
|2026-03-16 15:51:...|6294696619329796025|     NULL|               true|
+--------------------+-------------------+---------+-------------------+



In [5]:
df = spark.read.format("iceberg").load("lakehouse_prod.bronze_db.bronze_country_iso_profiles")
df.show()

+----+----+-------+-------------------+-------------------+--------------------+
|iso2|iso3|iso_num|            country|     country_common|       Ingested_Time|
+----+----+-------+-------------------+-------------------+--------------------+
|  AF| AFG|      4|        Afghanistan|        Afghanistan|2026-03-17 15:07:...|
|  AX| ALA|    248|      Aland Islands|      Aland Islands|2026-03-17 15:07:...|
|  AL| ALB|      8|            Albania|            Albania|2026-03-17 15:07:...|
|  DZ| DZA|     12|            Algeria|            Algeria|2026-03-17 15:07:...|
|  AS| ASM|     16|     American Samoa|     American Samoa|2026-03-17 15:07:...|
|  AD| AND|     20|            Andorra|            Andorra|2026-03-17 15:07:...|
|  AO| AGO|     24|             Angola|             Angola|2026-03-17 15:07:...|
|  AI| AIA|    660|           Anguilla|           Anguilla|2026-03-17 15:07:...|
|  AQ| ATA|     10|         Antarctica|         Antarctica|2026-03-17 15:07:...|
|  AG| ATG|     28|Antigua a

In [6]:
df.printSchema()

root
 |-- iso2: string (nullable = true)
 |-- iso3: string (nullable = true)
 |-- iso_num: long (nullable = true)
 |-- country: string (nullable = true)
 |-- country_common: string (nullable = true)
 |-- Ingested_Time: timestamp (nullable = true)



In [11]:
transformed_df = df\
  .withColumnRenamed('country', 'COUNTRY_NAME')\
  .withColumnRenamed('country_common', 'COUNTRY_FULL_NAME')\
  .withColumnRenamed('iso2', 'COUNTRY_ISO_CODE_2')\
  .withColumnRenamed('iso3', 'COUNTRY_ISO_CODE_3')\
  .withColumnRenamed('iso_num', 'COUNTRY_NUM_CODE')\
  .withColumnRenamed('Ingested_Time', 'INGESTED_TIME')\


In [9]:
transformed_df = transformed_df.withColumn("COUNTRY_KEY", concat(lit("ID-"), col("COUNTRY_ISO_CODE_3")))
transformed_df.select(
    "COUNTRY_KEY",
    "COUNTRY_NAME",
    "COUNTRY_FULL_NAME",
    "COUNTRY_ISO_CODE_2",
    "COUNTRY_ISO_CODE_3",
    "COUNTRY_NUM_CODE",
).show()

+-----------+-------------------+-------------------+------------------+------------------+----------------+
|COUNTRY_KEY|       COUNTRY_NAME|  COUNTRY_FULL_NAME|COUNTRY_ISO_CODE_2|COUNTRY_ISO_CODE_3|COUNTRY_NUM_CODE|
+-----------+-------------------+-------------------+------------------+------------------+----------------+
|     ID-AFG|        Afghanistan|        Afghanistan|                AF|               AFG|               4|
|     ID-ALA|      Aland Islands|      Aland Islands|                AX|               ALA|             248|
|     ID-ALB|            Albania|            Albania|                AL|               ALB|               8|
|     ID-DZA|            Algeria|            Algeria|                DZ|               DZA|              12|
|     ID-ASM|     American Samoa|     American Samoa|                AS|               ASM|              16|
|     ID-AND|            Andorra|            Andorra|                AD|               AND|              20|
|     ID-AGO|      

In [ ]:
        ingested_time = datetime.now(ZoneInfo("Asia/Bangkok"))
        extracted_df["Ingested_Time"] = ingested_time

In [12]:
df = spark.read.format("iceberg").load("lakehouse_prod.bronze_db.bronze_country_iso_profiles")

transformed_df = df\
  .withColumnRenamed('country', 'COUNTRY_NAME')\
  .withColumnRenamed('country_common', 'COUNTRY_FULL_NAME')\
  .withColumnRenamed('iso2', 'COUNTRY_ISO_CODE_2')\
  .withColumnRenamed('iso3', 'COUNTRY_ISO_CODE_3')\
  .withColumnRenamed('iso_num', 'COUNTRY_NUM_CODE')\
  .withColumnRenamed('Ingested_Time', 'INGESTED_TIME')\

ingested_time = datetime.now(ZoneInfo("Asia/Bangkok"))
transformed_df["PROCESS_TIME"] = ingested_time
transformed_df = transformed_df.withColumn(
    "COUNTRY_KEY",
    lpad(col("COUNTRY_NUM_CODE").cast("string"), 4, "0")
)
transformed_df.select(
    "COUNTRY_KEY",
    "COUNTRY_NAME",
    "COUNTRY_FULL_NAME",
    "COUNTRY_ISO_CODE_2",
    "COUNTRY_ISO_CODE_3",
    "COUNTRY_NUM_CODE",
).show()

+-----------+-------------------+-------------------+------------------+------------------+----------------+
|COUNTRY_KEY|       COUNTRY_NAME|  COUNTRY_FULL_NAME|COUNTRY_ISO_CODE_2|COUNTRY_ISO_CODE_3|COUNTRY_NUM_CODE|
+-----------+-------------------+-------------------+------------------+------------------+----------------+
|       0004|        Afghanistan|        Afghanistan|                AF|               AFG|               4|
|       0248|      Aland Islands|      Aland Islands|                AX|               ALA|             248|
|       0008|            Albania|            Albania|                AL|               ALB|               8|
|       0012|            Algeria|            Algeria|                DZ|               DZA|              12|
|       0016|     American Samoa|     American Samoa|                AS|               ASM|              16|
|       0020|            Andorra|            Andorra|                AD|               AND|              20|
|       0024|      

In [ ]:
transformed_df.

In [35]:
transformed_df.select(
    "COUNTRY_KEY",
    "COUNTRY_NAME",
    "COUNTRY_FULL_NAME",
    "COUNTRY_ISO_CODE_2",
    "COUNTRY_ISO_CODE_3",
    "COUNTRY_NUM_CODE",
).show()

+-----------+-------------------+-------------------+------------------+------------------+----------------+
|COUNTRY_KEY|       COUNTRY_NAME|  COUNTRY_FULL_NAME|COUNTRY_ISO_CODE_2|COUNTRY_ISO_CODE_3|COUNTRY_NUM_CODE|
+-----------+-------------------+-------------------+------------------+------------------+----------------+
|     ID-AFG|        Afghanistan|        Afghanistan|                AF|               AFG|               4|
|     ID-ALA|      Aland Islands|      Aland Islands|                AX|               ALA|             248|
|     ID-ALB|            Albania|            Albania|                AL|               ALB|               8|
|     ID-DZA|            Algeria|            Algeria|                DZ|               DZA|              12|
|     ID-ASM|     American Samoa|     American Samoa|                AS|               ASM|              16|
|     ID-AND|            Andorra|            Andorra|                AD|               AND|              20|
|     ID-AGO|      

In [3]:
spark.sql("""
SELECT * FROM lakehouse_prod.bronze_db.bronze_country_iso_profiles LIMIT 5;
""").show()

+----+----+-------+--------------+--------------+--------------------+
|iso2|iso3|iso_num|       country|country_common|       Ingested_Time|
+----+----+-------+--------------+--------------+--------------------+
|  AF| AFG|      4|   Afghanistan|   Afghanistan|2026-03-19 15:02:...|
|  AX| ALA|    248| Aland Islands| Aland Islands|2026-03-19 15:02:...|
|  AL| ALB|      8|       Albania|       Albania|2026-03-19 15:02:...|
|  DZ| DZA|     12|       Algeria|       Algeria|2026-03-19 15:02:...|
|  AS| ASM|     16|American Samoa|American Samoa|2026-03-19 15:02:...|
+----+----+-------+--------------+--------------+--------------------+



In [4]:
spark.stop()